# DOF and Vmode investigator

In [ ]:
import numpy as np
from lsst.ts.ofc.state_estimator import StateEstimator
from lsst.ts.ofc.controllers.oic_controller import OICController
from lsst.ts.ofc.ofc import OFC
from lsst.ts.ofc import OFCData

# Set up StateEstimator and OICController

# --- Compute vmodes from dof_state ---
ofcData = OFCData()
ofcData.configure_controller()
await ofcData.configure_instrument("lsst")

ofcData.xref = "0"

oic_controller = OICController(ofcData)

state_estimator = StateEstimator(ofcData)
ofc = OFC(ofcData)


In [ ]:
all_labels = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
     'Cam dz', 'Cam dx', 'Cam dy', 'Cam rx', 'Cam ry',
     'M1M3-1', 'M1M3-2', 'M1M3-3', 'M1M3-4', 'M1M3-5',
     'M1M3-6', 'M1M3-7', 'M1M3-8', 'M1M3-9', 'M1M3-10',
     'M1M3-11', 'M1M3-12', 'M1M3-13', 'M1M3-14', 'M1M3-15',
     'M1M3-16', 'M1M3-17', 'M1M3-18', 'M1M3-19', 'M1M3-20',
     'M2-1', 'M2-2', 'M2-3', 'M2-4', 'M2-5',
     'M2-6', 'M2-7', 'M2-8', 'M2-9', 'M2-10',
     'M2-11', 'M2-12', 'M2-13', 'M2-14', 'M2-15',
     'M2-16', 'M2-17', 'M2-18', 'M2-19', 'M2-20'
     ]

vmode_labels = ['Vmode1\nM2 tilts -rx-ry', 'Vmode2\nM2 tilts -rx+ry', 
                'Vmode3\nCam tilts -rx+ry', 'Vmode4\nCam tilts rx+ry',
                'Vmode5\nZ4-Focus', 'Vmode6\nZ5-Astig-Oblique',
                'Vmode7\nZ6-Astig-Vert', 'Vmode8\nZ7-Coma-Vert',
                'Vmode9\nZ8-Coma-Horiz', 'Vmode10\nZ9-Trefoil-Vert',
                'Vmode11\nZ10-Trefoil-Oblique', 'Vmode12\nZ11-Spherical']


In [ ]:
controller = ofc.controller

In [ ]:
controller.calculate_pid_step?

In [ ]:
dummy_zernikes = np.zeros([8,19])
for i in range(8):
    dummy_zernikes[i,0] = 5.0
sensor_names = ['R00_SW0', 'R00_SW1', 'R04_SW0', 'R04_SW1', 'R40_SW0', 'R40_SW1', 'R44_SW0', 'R44_SW1']
sensor_ids = [191, 192, 195, 196, 199, 200, 203, 204]
rotation_angle = 0.0
filter_name = 'R'
DOF_state = state_estimator.dof_state(filter_name, dummy_zernikes, sensor_names, rotation_angle)
print(DOF_state.shape)
vmodes = state_estimator.get_vmodes_from_dofs(DOF_state)[0:12]

print("DOF name\t DOF\t \tVmode name\t Vmode")
indices = list(range(0, 17)) + list(range(30,35))
#indices = list(range(22))
for i in indices:
    if i < 12:
        print(f"{all_labels[i]}\t{DOF_state[i]:10.4f}\t\tVmode{i+1}\t\t{vmodes[i]:8.3g}")
    else:
        print(f"{all_labels[i]}\t{DOF_state[i]:10.4f}")
    

In [ ]:
step = ofc.controller.control_step(filter_name, DOF_state, sensor_names, control_vmodes=False)
print(len(step))
print(step)
step = ofc.controller.control_step(filter_name, DOF_state, sensor_names, control_vmodes=True)
print(len(step))
print(step)

In [ ]:
# from 2026032500450
DOF_state = np.array([-7.69525284e+01, -6.10639338e-02,  2.36265858e-01, -9.13041173e-04,
       -3.64083977e-04, -3.08249989e+01,  7.34166702e-03, -2.81803659e-02,
       -1.04056016e-03, -1.91889829e-03,  2.92799535e-01, -3.08560693e-01,
        3.75287499e-01,  8.33104581e-03,  1.29503025e-02, -1.14993641e-01,
        3.93797551e-01,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  4.43675402e-01, -3.84422576e-01,
        1.76382587e-02,  3.38663001e-02,  2.41486633e-01,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00])

In [ ]:
# This is a key verification.
sensor_names = ['R00_SW0', 'R00_SW1', 'R04_SW0', 'R04_SW1', 'R40_SW0', 'R40_SW1', 'R44_SW0', 'R44_SW1']
filter_name = 'R'

step = ofc.controller.control_step(filter_name, DOF_state, sensor_names, control_vmodes=False)
print(len(step))
print(step)
calc_step = -0.18 * DOF_state + 0.022 * ofc.controller.integral
print(len(calc_step))
print(calc_step)
step == calc_step

In [ ]:
# From logs:
array([ 6.20842627e+01,  1.85024814e-02,  4.63174374e-02,  1.37509971e-03,
        6.66167483e-04, -8.38304926e+01, -2.22732106e-03, -5.52496989e-03,
        6.10750601e-04,  3.53968372e-04, -6.51272612e-04,  1.38463769e-02,
       -1.96929409e-01,  1.48467743e-02, -1.59776006e-02,  3.24834211e-02,
        7.71502726e-02, -2.01669203e-02,  2.99671735e-02, -1.72248928e-02,
       -7.42885321e-03, -1.29962377e-01])

In [ ]:
dict(
    m2HexPos=m2_hexapod,
    camHexPos=cam_hexapod,
    M1M3Bend=m1m3_bending,
    M2Bend=m2_bending,
)

In [ ]:
ofcData = OFCData()
ofcData.configure_controller()
await ofcData.configure_instrument("lsst")

ofcData.xref = "0"



# Restrict to standard_22 DOFs (5+5+7+5 = 22)
m2_hexapod = np.ones(5, dtype=bool)
cam_hexapod = np.ones(5, dtype=bool)
m1m3_bending = np.zeros(20, dtype=bool)
m2_bending = np.zeros(20, dtype=bool)
m1m3_bending[:7] = True
m2_bending[:5] = True
ofcData.comp_dof_idx = dict(
    m2HexPos=m2_hexapod,
    camHexPos=cam_hexapod,
    M1M3Bend=m1m3_bending,
    M2Bend=m2_bending,
)

state_estimator = StateEstimator(ofcData)
oic_controller = OICController(ofcData)

In [ ]:
ofc.calculate_corrections(dummy_zernikes, sensor_ids, filter_name, rotation_angle)

In [ ]:
# Vmode correction

vm = np.array([-4.72559362e-05,  4.13626351e-05,  7.38566873e-04,  8.25954557e-04,
       -7.84385725e-02, -2.78173216e-02, -8.72165269e-03,  2.49910439e-03,
        1.04508069e-02, -2.80515497e-01, -5.30392364e-01, -4.42246410e-01,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00])

In [ ]:
ofcData = OFCData()
ofcData.configure_controller()
await ofcData.configure_instrument("lsst")

# Restrict to standard_22 DOFs (5+5+7+5 = 22)
m2_hexapod = np.ones(5, dtype=bool)
cam_hexapod = np.ones(5, dtype=bool)
m1m3_bending = np.zeros(20, dtype=bool)
m2_bending = np.zeros(20, dtype=bool)
m1m3_bending[:7] = True
m2_bending[:5] = True
ofcData.comp_dof_idx = dict(
    m2HexPos=m2_hexapod,
    camHexPos=cam_hexapod,
    M1M3Bend=m1m3_bending,
    M2Bend=m2_bending,
)

state_estimator = StateEstimator(ofcData)


In [ ]:
state_estimator.get_dofs_from_vmodes(vm)